# MemoryBandit Cross-Region PAC (Smoke Test)

This notebook follows the state-space PAC workflow described in Soulat et al. (2022) and implemented in `somata`:

1. Fit oscillator models to two regions.
2. Use **slow phase** from Region A and **fast amplitude** from Region B.
3. Fit constrained PAC regression (no surrogate/permutation data).
4. Fit optional AR(1) smoothing over windowed PAC coefficients (dSSP-style).

The synthetic data generation here is only a smoke test scaffold. Replace `region_a` / `region_b` with your MemoryBandit channels in the same shape.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from importlib.util import find_spec
from IPython.display import display, Image
from mne_connectivity import seed_target_indices

from mv_granger_utils import (
    load_behavior_tables,
    preprocess_learn_df,
    preprocess_combined_df,
)

sys.path.append('/Users/salmanqasim/Documents/GitRepos/LFPAnalysis')
from LFPAnalysis import oscillation_utils

ROOT = Path.cwd().resolve()
if (ROOT / "somata").exists():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / "somata").exists():
    sys.path.insert(0, str(ROOT.parent))

if find_spec("cmdstanpy") is None:
    raise ImportError(
        "cmdstanpy is required for fit_pac_regression. Install cmdstanpy and CmdStan first."
    )

from cmdstanpy import cmdstan_path
try:
    _ = cmdstan_path()
except ValueError as exc:
    raise RuntimeError(
        "CmdStan is not installed. Install CmdStan before running this notebook."
    ) from exc

from somata.oscillator_search import IterativeOscillatorModel
from somata.pac import fit_pac_regression, kmod, phimod, optimize_arp, mvar_ssm

rng = np.random.default_rng(7)
SMOKE_PLOT_DIR = Path.cwd() / "smoke_outputs"
SMOKE_PLOT_DIR.mkdir(parents=True, exist_ok=True)

# MemoryBandit smoke configuration (GC-style subject loading)
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
day = 'Day1'
analysis_event = 'feedback_start'
phase_roi = 'HPC'
amp_roi = 'OFC'
SMOKE_MAX_PAIRS = 1
SMOKE_MAX_TRIALS = 20

# Demographics-based subject filtering (same pattern as GC notebook)
subjects = []
subj_df = pd.DataFrame()
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
if os.path.exists(demo_file):
    demo_data = pd.read_excel(demo_file, engine='openpyxl')
    subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
    subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]
    subjects = sorted(subj_df['MSSMCode'].dropna().astype(str).unique().tolist())

SMOKE_SUBJECT = ('MS036' if 'MS036' in subjects else (subjects[0] if len(subjects) > 0 else None))

# Behavior table loading (for parity with existing notebook stack)
learn_df = pd.DataFrame()
combined_df = pd.DataFrame()
try:
    learn_df_raw, combined_df_raw = load_behavior_tables(load_dir)
    learn_df = preprocess_learn_df(learn_df_raw)
    combined_df = preprocess_combined_df(combined_df_raw)
except Exception as exc:
    print(f'Behavior tables unavailable for smoke context: {exc}')

print('Subjects available:', len(subjects))
print('Selected smoke subject:', SMOKE_SUBJECT)
print('Smoke pair:', f'{phase_roi} -> {amp_roi}')
print('Event/day:', analysis_event, day)
print('SMOKE_MAX_PAIRS:', SMOKE_MAX_PAIRS, '| SMOKE_MAX_TRIALS:', SMOKE_MAX_TRIALS)



In [ ]:
def make_windows(x, window_length, window_stride):
    starts = np.arange(0, len(x) - window_length + 1, window_stride)
    windows = [x[s : s + window_length] for s in starts]
    return starts, windows


def generate_cross_region_pac_data(
    fs=100,
    seconds=36,
    f_slow=6.0,
    f_fast=30.0,
    k_mod=0.45,
    phi_mod=0.4,
    noise=0.35,
    rng=np.random.default_rng(7),
):
    t = np.arange(0, seconds, 1 / fs)
    slow_phase = 2 * np.pi * f_slow * t
    slow = np.sin(slow_phase)
    region_a = slow + 0.2 * np.sin(2 * np.pi * 12.0 * t) + noise * rng.standard_normal(t.shape[0])
    fast_envelope = 1.0 + k_mod * np.cos(slow_phase - phi_mod)
    fast = fast_envelope * np.sin(2 * np.pi * f_fast * t + 0.2)
    region_b = fast + 0.2 * np.sin(2 * np.pi * 8.0 * t) + noise * rng.standard_normal(t.shape[0])
    truth = {"k_mod": k_mod, "phi_mod": phi_mod, "f_slow": f_slow, "f_fast": f_fast}
    return t, region_a, region_b, truth


def extract_phase_and_amplitude(region_a, region_b, fs):
    iosc_a = IterativeOscillatorModel(region_a, fs)
    iosc_a.iterate()
    osc_a = iosc_a.get_knee_osc()
    xa = osc_a.kalman_filt_smooth(y=region_a[None, :], return_dict=True)["x_t_n"]

    iosc_b = IterativeOscillatorModel(region_b, fs)
    iosc_b.iterate()
    osc_b = iosc_b.get_knee_osc()
    xb = osc_b.kalman_filt_smooth(y=region_b[None, :], return_dict=True)["x_t_n"]

    phase_a = np.arctan2(xa[1, 1:], xa[0, 1:])
    amp_b = np.sqrt(np.sum(xb[:, 1:] ** 2, axis=0))
    return phase_a, amp_b, osc_a, osc_b


def fit_pac_map(phase, amp, chains=1, iter_warmup=120, iter_sampling=120, seed=11):
    draws = fit_pac_regression(
        phase,
        amp,
        verbose=False,
        chains=chains,
        parallel_chains=chains,
        iter_warmup=iter_warmup,
        iter_sampling=iter_sampling,
        seed=seed,
        refresh=0,
    )
    beta = draws.mean(axis=0)
    return draws, beta


def load_smoke_subject_epochs(subj_id, base_dir, day, event):
    if subj_id is None:
        return None, None, None, None, 'No smoke subject available from demographics filtering.'

    epochs_file = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}/{event}-epo.fif'
    elec_df_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df'

    if not os.path.exists(epochs_file):
        return None, None, None, None, f'Missing epochs file: {epochs_file}'
    if not os.path.exists(elec_df_path):
        return None, None, None, None, f'Missing electrode table: {elec_df_path}'

    epochs_reref = mne.read_epochs(epochs_file, preload=True, verbose=False)
    elec_df = pd.read_csv(elec_df_path)
    sfreq = float(epochs_reref.info['sfreq'])
    times = np.asarray(epochs_reref.times, dtype=float)
    return epochs_reref, elec_df, sfreq, times, None


def run_synthetic_fallback_trials(fs=100):
    t, region_a, region_b, truth = generate_cross_region_pac_data(fs=fs, seconds=36, rng=rng)
    phase_a, amp_b, _, _ = extract_phase_and_amplitude(region_a, region_b, fs)

    window_sec = 6
    stride_sec = 6
    window_len = int(window_sec * fs)
    window_stride = int(stride_sec * fs)

    region_a_aligned = region_a[1:]
    region_b_aligned = region_b[1:]
    t_aligned = t[1:]

    starts, phase_w = make_windows(phase_a, window_len, window_stride)
    _, amp_w = make_windows(amp_b, window_len, window_stride)
    _, region_a_trials = make_windows(region_a_aligned, window_len, window_stride)
    _, region_b_trials = make_windows(region_b_aligned, window_len, window_stride)
    _, times_trials = make_windows(t_aligned, window_len, window_stride)

    trial_records = []
    trial_plot_payloads = {}

    for i, (p_win, a_win) in enumerate(zip(phase_w, amp_w)):
        metadata = {
            'participant': 'SYNTH001',
            'event': 'synthetic_event',
            'phase_roi': 'RegionA',
            'amp_roi': 'RegionB',
            'phase_channel': 'region_a',
            'amp_channel': 'region_b',
            'hemi': 'n/a',
            'trial_idx': int(i),
            'alignment_valid': True,
        }

        fit_ok = False
        err_msg = None
        beta_i = np.array([np.nan, np.nan, np.nan], dtype=float)
        k_i = np.nan
        phi_i = np.nan

        try:
            _, beta_i = fit_pac_map(p_win, a_win, iter_warmup=120, iter_sampling=120, seed=100 + i)
            k_i = float(kmod(beta_i[0], beta_i[1], beta_i[2]))
            phi_i = float(phimod(beta_i[1], beta_i[2]))
            fit_ok = bool(np.isfinite(k_i))
        except Exception as exc:
            err_msg = str(exc)

        rec = {
            **metadata,
            'somata_fit_ok': bool(fit_ok),
            'somata_error': err_msg,
            'somata_beta0': float(beta_i[0]) if np.isfinite(beta_i[0]) else np.nan,
            'somata_beta1': float(beta_i[1]) if np.isfinite(beta_i[1]) else np.nan,
            'somata_beta2': float(beta_i[2]) if np.isfinite(beta_i[2]) else np.nan,
            'somata_kmod': float(k_i) if np.isfinite(k_i) else np.nan,
            'somata_phi_mod': float(phi_i) if np.isfinite(phi_i) else np.nan,
        }
        trial_records.append(rec)

        if bool(fit_ok) and np.isfinite(k_i):
            trial_plot_payloads[i] = {
                **metadata,
                'phase_epoch': np.asarray(region_a_trials[i], dtype=float),
                'amp_epoch': np.asarray(region_b_trials[i], dtype=float),
                'phase_vec': np.asarray(p_win, dtype=float),
                'amp_vec': np.asarray(a_win, dtype=float),
                'times': np.asarray(times_trials[i], dtype=float),
                'sfreq': float(fs),
                'somata_beta0': float(beta_i[0]),
                'somata_beta1': float(beta_i[1]),
                'somata_beta2': float(beta_i[2]),
                'somata_kmod': float(k_i),
                'somata_phi_mod': float(phi_i),
            }

    context = {
        'mode': 'synthetic_fallback',
        't': t,
        'phase_signal': region_a,
        'amp_signal': region_b,
        'truth': truth,
        'phase_w': phase_w,
        'starts': starts,
    }
    return trial_records, trial_plot_payloads, context


def plot_single_high_trial(high_trial, out_dir):
    phase_epoch = np.asarray(high_trial['phase_epoch'], dtype=float)
    amp_epoch = np.asarray(high_trial['amp_epoch'], dtype=float)
    phase_vec = np.asarray(high_trial['phase_vec'], dtype=float)
    amp_vec = np.asarray(high_trial['amp_vec'], dtype=float)
    times = np.asarray(high_trial['times'], dtype=float)

    if times.shape[0] != phase_epoch.shape[0]:
        times = np.arange(phase_epoch.shape[0]) / float(high_trial['sfreq'])

    times_amp = times if amp_vec.shape[0] == times.shape[0] else np.linspace(times[0], times[-1], amp_vec.shape[0])

    beta0 = float(high_trial['somata_beta0'])
    beta1 = float(high_trial['somata_beta1'])
    beta2 = float(high_trial['somata_beta2'])

    phase_bins = np.linspace(-np.pi, np.pi, 19)
    phase_centers = (phase_bins[:-1] + phase_bins[1:]) / 2.0
    phase_idx = np.digitize(phase_vec, phase_bins) - 1
    phase_profile = np.array(
        [np.nanmean(amp_vec[phase_idx == b]) if np.any(phase_idx == b) else np.nan for b in range(len(phase_bins) - 1)],
        dtype=float,
    )

    phi_grid = np.linspace(-np.pi, np.pi, 400)
    amp_fit = beta0 + beta1 * np.cos(phi_grid) + beta2 * np.sin(phi_grid)

    fig, axes = plt.subplots(3, 1, figsize=(12, 10), constrained_layout=True)

    axes[0].plot(times, phase_epoch, color='#2e86de', linewidth=1.2)
    axes[0].set_title('High-scoring trial: raw phase-channel epoch')
    axes[0].set_ylabel('Phase channel LFP')
    axes[0].set_xlabel('Time (s)')

    axes[1].plot(times, amp_epoch, color='#e67e22', linewidth=1.1, label='Raw amp-channel epoch')
    ax2 = axes[1].twinx()
    ax2.plot(times_amp, amp_vec, color='#16a085', linewidth=1.1, alpha=0.9, label='SOMATA extracted amplitude')
    axes[1].set_title('Raw amplitude channel + extracted amplitude trace')
    axes[1].set_ylabel('Amp channel LFP')
    ax2.set_ylabel('Extracted amplitude')
    axes[1].set_xlabel('Time (s)')
    h1, l1 = axes[1].get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    axes[1].legend(h1 + h2, l1 + l2, loc='upper right')

    axes[2].scatter(phase_vec, amp_vec, s=10, alpha=0.25, color='#7f8c8d', label='Samples')
    axes[2].plot(phase_centers, phase_profile, marker='o', color='#8e44ad', linewidth=1.5, label='Binned mean amplitude')
    axes[2].plot(phi_grid, amp_fit, color='#c0392b', linewidth=2.0, label='Fitted constrained curve')
    axes[2].set_title('Phase-amplitude profile with fitted SOMATA PAC curve')
    axes[2].set_xlabel('Phase (rad)')
    axes[2].set_ylabel('Amplitude')
    axes[2].legend(loc='best')
    axes[2].text(
        0.01,
        0.98,
        f"kmod={high_trial['somata_kmod']:.4f} | phi_mod={high_trial['somata_phi_mod']:.4f} rad",
        transform=axes[2].transAxes,
        va='top',
        ha='left',
        fontsize=10,
        bbox=dict(facecolor='white', alpha=0.85, edgecolor='none'),
    )

    fig.suptitle(
        f"{high_trial['participant']} | {high_trial['event']} | "
        f"{high_trial['phase_roi']}->{high_trial['amp_roi']} | "
        f"{high_trial['phase_channel']}->{high_trial['amp_channel']} | "
        f"trial {high_trial['trial_idx'] + 1}",
        fontsize=12,
    )

    file_name = (
        f"somata_hightrial_{high_trial['participant']}_{high_trial['event']}_"
        f"{high_trial['phase_roi']}-{high_trial['amp_roi']}_"
        f"{high_trial['phase_channel']}-{high_trial['amp_channel']}_"
        f"trial{high_trial['trial_idx'] + 1}.png"
    )
    safe_name = file_name.replace('/', '-').replace(' ', '')
    out_file = Path(out_dir) / safe_name
    fig.savefig(out_file, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return out_file



In [ ]:
run_mode = 'real_subject'
fallback_reason = None

trial_records = []
trial_plot_payloads = {}
plot_context = {}
used_pairs = []

# Try real-subject path first
epochs_reref, elec_df, sfreq_real, times_real, load_err = load_smoke_subject_epochs(
    subj_id=SMOKE_SUBJECT,
    base_dir=base_dir,
    day=day,
    event=analysis_event,
)

if load_err is None:
    seed_target_df = oscillation_utils.make_seed_target_df(elec_df, epochs_reref, phase_roi, amp_roi)
    seed_target_df = seed_target_df.rename(columns={'seed': 'phase', 'target': 'amp'})

    pair_counter = 0
    for hemi in ['l', 'r']:
        if hemi not in seed_target_df.index.tolist():
            continue

        seed_to_target = seed_target_indices(seed_target_df['phase'][hemi], seed_target_df['amp'][hemi])
        n_pairs = len(seed_to_target[0])
        if n_pairs == 0:
            continue

        for pair_idx in range(n_pairs):
            if pair_counter >= SMOKE_MAX_PAIRS:
                break

            ph_ch_idx = seed_to_target[0][pair_idx]
            amp_ch_idx = seed_target_df['amp'][hemi][seed_to_target[1][pair_idx]]

            ph_ch_name = epochs_reref.ch_names[ph_ch_idx]
            amp_ch_name = epochs_reref.ch_names[amp_ch_idx]

            ph_sig = np.asarray(epochs_reref.copy().pick(ph_ch_name).get_data(), dtype=float)
            amp_sig = np.asarray(epochs_reref.copy().pick(amp_ch_name).get_data(), dtype=float)
            if ph_sig.ndim == 1:
                ph_sig = ph_sig[None, :]
            if amp_sig.ndim == 1:
                amp_sig = amp_sig[None, :]

            n_epochs = int(min(ph_sig.shape[0], amp_sig.shape[0], SMOKE_MAX_TRIALS))
            if n_epochs == 0:
                continue

            used_pairs.append({'hemi': hemi, 'phase_channel': ph_ch_name, 'amp_channel': amp_ch_name})

            # Save one pair's raw traces for context plot
            if 'times' not in plot_context:
                plot_context['times'] = np.asarray(times_real, dtype=float)
                plot_context['phase_signal'] = np.nanmean(ph_sig[:n_epochs], axis=0)
                plot_context['amp_signal'] = np.nanmean(amp_sig[:n_epochs], axis=0)

            for trial_idx in range(n_epochs):
                phase_epoch = np.asarray(ph_sig[trial_idx], dtype=float)
                amp_epoch = np.asarray(amp_sig[trial_idx], dtype=float)

                metadata = {
                    'participant': str(SMOKE_SUBJECT),
                    'event': str(analysis_event),
                    'phase_roi': str(phase_roi),
                    'amp_roi': str(amp_roi),
                    'phase_channel': str(ph_ch_name),
                    'amp_channel': str(amp_ch_name),
                    'hemi': str(hemi),
                    'trial_idx': int(trial_idx),
                    'alignment_valid': True,
                }

                fit_ok = False
                err_msg = None
                beta_i = np.array([np.nan, np.nan, np.nan], dtype=float)
                k_i = np.nan
                phi_i = np.nan
                phase_vec = np.array([], dtype=float)
                amp_vec = np.array([], dtype=float)

                try:
                    phase_vec, amp_vec, _, _ = extract_phase_and_amplitude(phase_epoch, amp_epoch, float(sfreq_real))
                    _, beta_i = fit_pac_map(phase_vec, amp_vec, iter_warmup=120, iter_sampling=120, seed=100 + trial_idx)
                    k_i = float(kmod(beta_i[0], beta_i[1], beta_i[2]))
                    phi_i = float(phimod(beta_i[1], beta_i[2]))
                    fit_ok = bool(np.isfinite(k_i))
                except Exception as exc:
                    err_msg = str(exc)

                rec = {
                    **metadata,
                    'somata_fit_ok': bool(fit_ok),
                    'somata_error': err_msg,
                    'somata_beta0': float(beta_i[0]) if np.isfinite(beta_i[0]) else np.nan,
                    'somata_beta1': float(beta_i[1]) if np.isfinite(beta_i[1]) else np.nan,
                    'somata_beta2': float(beta_i[2]) if np.isfinite(beta_i[2]) else np.nan,
                    'somata_kmod': float(k_i) if np.isfinite(k_i) else np.nan,
                    'somata_phi_mod': float(phi_i) if np.isfinite(phi_i) else np.nan,
                }
                trial_records.append(rec)

                if bool(fit_ok) and np.isfinite(k_i):
                    time_vec = np.asarray(times_real, dtype=float)
                    trial_plot_payloads[(metadata['phase_channel'], metadata['amp_channel'], metadata['trial_idx'])] = {
                        **metadata,
                        'phase_epoch': phase_epoch,
                        'amp_epoch': amp_epoch,
                        'phase_vec': np.asarray(phase_vec, dtype=float),
                        'amp_vec': np.asarray(amp_vec, dtype=float),
                        'times': time_vec,
                        'sfreq': float(sfreq_real),
                        'somata_beta0': float(beta_i[0]),
                        'somata_beta1': float(beta_i[1]),
                        'somata_beta2': float(beta_i[2]),
                        'somata_kmod': float(k_i),
                        'somata_phi_mod': float(phi_i),
                    }

            pair_counter += 1

        if pair_counter >= SMOKE_MAX_PAIRS:
            break

    if len(trial_records) > 0:
        run_mode = 'real_subject'
    else:
        fallback_reason = f'No valid seed-target pairs or trials for {phase_roi}->{amp_roi} in subject {SMOKE_SUBJECT}.'
else:
    fallback_reason = load_err

# Fallback to synthetic if real path unavailable
if run_mode != 'real_subject':
    print('Real-subject path unavailable; switching to synthetic fallback.')
    print('Reason:', fallback_reason)
    trial_records, trial_plot_payloads, synth_context = run_synthetic_fallback_trials(fs=100)
    plot_context.update(synth_context)

trial_results_df = pd.DataFrame(trial_records)

required_cols = [
    'participant', 'event', 'phase_roi', 'amp_roi', 'phase_channel', 'amp_channel', 'hemi',
    'trial_idx', 'alignment_valid', 'somata_fit_ok', 'somata_error',
    'somata_beta0', 'somata_beta1', 'somata_beta2', 'somata_kmod', 'somata_phi_mod',
]
for c in required_cols:
    if c not in trial_results_df.columns:
        trial_results_df[c] = np.nan

valid_mask = (
    trial_results_df['somata_fit_ok'].astype(bool)
    & np.isfinite(trial_results_df['somata_kmod'].to_numpy(dtype=float))
    & trial_results_df['alignment_valid'].astype(bool)
)

trial_results_df['trial_pac_z'] = np.nan
if valid_mask.any():
    vals = trial_results_df.loc[valid_mask, 'somata_kmod'].to_numpy(dtype=float)
    std = np.nanstd(vals)
    zvals = (vals - np.nanmean(vals)) / std if np.isfinite(std) and std > 0 else np.zeros(vals.shape[0], dtype=float)
    trial_results_df.loc[valid_mask, 'trial_pac_z'] = zvals

trial_results_df['is_global_high_trial'] = False
global_high_trial = None

if valid_mask.any():
    cand = trial_results_df.loc[valid_mask].copy()
    cand['trial_pac_z_sort'] = cand['trial_pac_z'].fillna(-np.inf)
    cand = cand.sort_values(
        by=[
            'somata_kmod', 'trial_pac_z_sort', 'participant', 'event',
            'phase_roi', 'amp_roi', 'phase_channel', 'amp_channel', 'trial_idx',
        ],
        ascending=[False, False, True, True, True, True, True, True, True],
        kind='mergesort',
    )
    best_row = cand.iloc[0]

    best_key = (
        str(best_row['phase_channel']),
        str(best_row['amp_channel']),
        int(best_row['trial_idx']),
    )

    row_match = (
        (trial_results_df['phase_channel'].astype(str) == str(best_row['phase_channel']))
        & (trial_results_df['amp_channel'].astype(str) == str(best_row['amp_channel']))
        & (trial_results_df['trial_idx'].astype(int) == int(best_row['trial_idx']))
    )
    trial_results_df.loc[row_match, 'is_global_high_trial'] = True

    global_high_trial = trial_plot_payloads.get(best_key, None)
    if global_high_trial is not None:
        global_high_trial['trial_pac_z'] = float(best_row['trial_pac_z'])

fit_df = trial_results_df[trial_results_df['somata_fit_ok'] & np.isfinite(trial_results_df['somata_beta0'])].copy()
fit_df = fit_df.sort_values(['participant', 'event', 'phase_channel', 'amp_channel', 'trial_idx'])

beta_map = np.empty((3, 0), dtype=float)
beta_smooth = np.empty((3, 0), dtype=float)
k_smooth_valid = np.array([], dtype=float)

if len(fit_df) > 0:
    beta_map = fit_df[['somata_beta0', 'somata_beta1', 'somata_beta2']].to_numpy(dtype=float).T

if beta_map.shape[1] >= 3:
    A, Q, R = optimize_arp(beta_map, p=1)
    ar_model = mvar_ssm(beta_map, A, Q, R)
    z = ar_model.kalman_filt_smooth(return_dict=True)
    beta_smooth = z['x_t_n'][:3, 1:]
    k_smooth_valid = kmod(beta_smooth[0], beta_smooth[1], beta_smooth[2])

plot_df = trial_results_df.sort_values(['participant', 'event', 'phase_channel', 'amp_channel', 'trial_idx']).copy()
x_idx = np.arange(len(plot_df), dtype=int)
k_raw_all = plot_df['somata_kmod'].to_numpy(dtype=float)
k_smooth_all = np.full(k_raw_all.shape[0], np.nan, dtype=float)
if len(fit_df) > 0 and k_smooth_valid.size == len(fit_df):
    fit_positions = fit_df.index.to_series().map({idx: pos for pos, idx in enumerate(plot_df.index)}).dropna().astype(int).to_numpy()
    k_smooth_all[fit_positions] = k_smooth_valid

print('run_mode:', run_mode)
print('SMOKE_SUBJECT:', SMOKE_SUBJECT)
print('Used pairs:', used_pairs if run_mode == 'real_subject' else 'synthetic fallback pair')
print('Trials attempted:', len(trial_results_df))
print('Trials successful:', int(trial_results_df['somata_fit_ok'].sum()))
trial_results_df.head()



In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(12, 10), sharex=False)

if run_mode == 'real_subject' and ('times' in plot_context):
    ax[0].plot(plot_context['times'], plot_context['phase_signal'], label='Mean phase-channel epoch', lw=1)
    ax[0].plot(plot_context['times'], plot_context['amp_signal'], label='Mean amp-channel epoch', lw=1, alpha=0.7)
    ax[0].set_title('Real-subject context signal (mean across smoke trials)')
    ax[0].set_xlabel('Time (s)')
else:
    ax[0].plot(plot_context['t'], plot_context['phase_signal'], label='Region A (phase source)', lw=1)
    ax[0].plot(plot_context['t'], plot_context['amp_signal'], label='Region B (amplitude source)', lw=1, alpha=0.7)
    ax[0].set_title('Synthetic fallback context signal')
    ax[0].set_xlabel('Time (s)')
ax[0].legend(loc='upper right')

ax[1].plot(x_idx, k_raw_all, marker='o', label='Raw trial Kmod')
if np.isfinite(k_smooth_all).any():
    ax[1].plot(x_idx, k_smooth_all, marker='o', label='Smoothed Kmod', alpha=0.9)
if run_mode == 'synthetic_fallback' and isinstance(plot_context.get('truth', None), dict):
    ax[1].axhline(plot_context['truth']['k_mod'], color='k', linestyle='--', linewidth=1, label='Truth')

if global_high_trial is not None:
    row_pos = plot_df.index[plot_df['is_global_high_trial']].tolist()
    if len(row_pos) > 0:
        hi_plot_pos = int(np.where(plot_df.index == row_pos[0])[0][0])
        if np.isfinite(k_raw_all[hi_plot_pos]):
            ax[1].scatter([hi_plot_pos], [k_raw_all[hi_plot_pos]], color='#c0392b', s=70, zorder=5, label='Global high trial')

ax[1].set_title('Trial-level SOMATA PAC strength')
ax[1].set_ylabel('Kmod')
ax[1].set_xlabel('Smoke trial order')
ax[1].legend(loc='upper right')

if beta_map.shape[1] > 0:
    ax[2].plot(np.arange(beta_map.shape[1]), beta_map[0], marker='o', label='beta0 (raw)')
if beta_smooth.shape[1] > 0:
    ax[2].plot(np.arange(beta_smooth.shape[1]), beta_smooth[0], marker='o', label='beta0 (smoothed)', alpha=0.9)
ax[2].set_title('SOMATA PAC intercept over ordered valid fits')
ax[2].set_xlabel('Valid-fit order')
ax[2].legend(loc='upper right')

plt.tight_layout()
plt.show()



In [ ]:
high_trial_plot_file = None

if global_high_trial is not None:
    high_trial_plot_file = plot_single_high_trial(global_high_trial, SMOKE_PLOT_DIR)
    print('Global high trial selected:')
    print(
        f"  participant={global_high_trial['participant']} | event={global_high_trial['event']} | "
        f"pair={global_high_trial['phase_roi']}->{global_high_trial['amp_roi']} | "
        f"channels={global_high_trial['phase_channel']}->{global_high_trial['amp_channel']} | "
        f"trial_idx={global_high_trial['trial_idx']}"
    )
    print(
        f"  somata_kmod={global_high_trial['somata_kmod']:.4f} | "
        f"trial_pac_z={global_high_trial.get('trial_pac_z', np.nan):.4f} | "
        f"phi_mod={global_high_trial['somata_phi_mod']:.4f}"
    )
    print('Saved high-trial plot:', high_trial_plot_file)
    display(Image(filename=str(high_trial_plot_file)))
else:
    print('No valid high-scoring trial found; skipping high-trial plotting.')

# Smoke validation
assert len(trial_results_df) > 0, 'No smoke trial rows were generated.'
assert (trial_results_df['somata_fit_ok'].sum() >= 1), 'Expected at least one successful SOMATA trial fit.'
assert trial_results_df['trial_idx'].between(0, max(int(trial_results_df['trial_idx'].max()), 0)).all(), 'Trial indices are out of bounds.'

if global_high_trial is not None:
    assert high_trial_plot_file is not None and Path(high_trial_plot_file).exists(), 'High-trial plot file was not created.'
    assert np.isfinite(global_high_trial['somata_kmod']), 'Selected high-trial Kmod is not finite.'

print('Smoke test passed.')



## Mapping To Your MemoryBandit Notebook

- Replace `region_a` with your slow-phase channel/region trace.
- Replace `region_b` with your fast-amplitude channel/region trace.
- Keep `extract_phase_and_amplitude(...)` and the windowed PAC block as-is.
- Keep surrogate/permutation testing disabled (none is used in this notebook).

If your event structure is trial-based, use trial windows directly instead of fixed `window_sec` / `stride_sec`.